In [1]:
%load_ext sql
%sql sqlite:///advanced_sql.db

Connecting to 'sqlite:///advanced_sql.db'

In [2]:
%%sql
DROP TABLE IF EXISTS sales;
CREATE TABLE sales (
    sale_date DATE,
    salesperson TEXT,
    region TEXT,
    amount INTEGER
);

INSERT INTO sales (sale_date, salesperson, region, amount) VALUES 
('2026-02-01', 'Alice', 'North', 500),
('2026-02-02', 'Alice', 'North', 800),
('2026-02-03', 'Alice', 'North', 300),
('2026-02-01', 'Bob', 'South', 1000),
('2026-02-02', 'Bob', 'South', 200),
('2026-02-03', 'Bob', 'South', 900),
('2026-02-01', 'Charlie', 'North', 750),
('2026-02-02', 'Charlie', 'North', 600);

Running query in 'sqlite:///advanced_sql.db'

8 rows affected.

++
||
++
++

In [3]:
%%sql
SELECT 
    sale_date,
    salesperson,
    region,
    amount,
    SUM(amount) OVER(PARTITION BY region) AS total_regional_sales
FROM sales
ORDER BY region, sale_date;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,region,amount,total_regional_sales
2026-02-01,Alice,North,500,2950
2026-02-01,Charlie,North,750,2950
2026-02-02,Alice,North,800,2950
2026-02-02,Charlie,North,600,2950
2026-02-03,Alice,North,300,2950
2026-02-01,Bob,South,1000,2100
2026-02-02,Bob,South,200,2100
2026-02-03,Bob,South,900,2100


In [13]:
%%sql
SELECT sale_date, salesperson, region, SUM(AMOUNT) OVER(PARTITION BY region) AS total_regional_sales
FROM sales
ORDER BY sale_date;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,region,total_regional_sales
2026-02-01,Alice,North,2950
2026-02-01,Charlie,North,2950
2026-02-01,Bob,South,2100
2026-02-02,Alice,North,2950
2026-02-02,Charlie,North,2950
2026-02-02,Bob,South,2100
2026-02-03,Alice,North,2950
2026-02-03,Bob,South,2100


In [15]:
%%sql
SELECT sale_date, salesperson, region, amount, sum(amount) over(partition by salesperson order by sale_date) as indiv_sales
FROM sales;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,region,amount,indiv_sales
2026-02-01,Alice,North,500,500
2026-02-02,Alice,North,800,1300
2026-02-03,Alice,North,300,1600
2026-02-01,Bob,South,1000,1000
2026-02-02,Bob,South,200,1200
2026-02-03,Bob,South,900,2100
2026-02-01,Charlie,North,750,750
2026-02-02,Charlie,North,600,1350


In [12]:
%%sql
SELECT sale_date, salesperson, region, amount, rank() over(partition by salesperson order by amount DESC) as rank
FROM sales
order by sale_date;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,region,amount,rank
2026-02-01,Alice,North,500,2
2026-02-01,Bob,South,1000,1
2026-02-01,Charlie,North,750,1
2026-02-02,Alice,North,800,1
2026-02-02,Bob,South,200,3
2026-02-02,Charlie,North,600,2
2026-02-03,Alice,North,300,3
2026-02-03,Bob,South,900,2


In [ ]:
%%sql
SELECT 
    salesperson,
    region,
    amount,
    RANK() OVER(PARTITION BY region ORDER BY amount DESC) as regional_rank
FROM sales;

In [16]:
%%sql
SELECT 
    sale_date,
    salesperson,
    amount,
    SUM(amount) OVER(
        PARTITION BY salesperson 
        ORDER BY sale_date
    ) AS running_total
FROM sales;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,amount,running_total
2026-02-01,Alice,500,500
2026-02-02,Alice,800,1300
2026-02-03,Alice,300,1600
2026-02-01,Bob,1000,1000
2026-02-02,Bob,200,1200
2026-02-03,Bob,900,2100
2026-02-01,Charlie,750,750
2026-02-02,Charlie,600,1350


In [17]:
%%sql
SELECT 
    sale_date,
    salesperson,
    amount AS todays_sales,
    LAG(amount, 1) OVER(PARTITION BY salesperson ORDER BY sale_date) AS yesterdays_sales,
    amount - LAG(amount, 1) OVER(PARTITION BY salesperson ORDER BY sale_date) AS daily_difference
FROM sales;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,todays_sales,yesterdays_sales,daily_difference
2026-02-01,Alice,500,None,None
2026-02-02,Alice,800,500,300
2026-02-03,Alice,300,800,-500
2026-02-01,Bob,1000,None,None
2026-02-02,Bob,200,1000,-800
2026-02-03,Bob,900,200,700
2026-02-01,Charlie,750,None,None
2026-02-02,Charlie,600,750,-150


In [32]:
%%sql
select sale_date, salesperson, region, amount as todays,
lag(amount, 2) over(partition by salesperson order by sale_date) as day_before,
lag(amount, 2) over(partition by salesperson order by sale_date) - amount as two_day_diff
FROM sales
order by region;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,region,todays,day_before,two_day_diff
2026-02-01,Alice,North,500,None,None
2026-02-02,Alice,North,800,None,None
2026-02-03,Alice,North,300,500,200
2026-02-01,Charlie,North,750,None,None
2026-02-02,Charlie,North,600,None,None
2026-02-01,Bob,South,1000,None,None
2026-02-02,Bob,South,200,None,None
2026-02-03,Bob,South,900,1000,100


In [18]:
%%sql
DROP TABLE IF EXISTS session_telemetry;

CREATE TABLE session_telemetry (
    log_id SERIAL PRIMARY KEY, -- In SQLite, use: log_id INTEGER PRIMARY KEY AUTOINCREMENT
    user_id VARCHAR(50),
    timestamp TIMESTAMP,
    ping_ms INTEGER
);

INSERT INTO session_telemetry (user_id, timestamp, ping_ms) VALUES
-- User 1: Normal, stable connection
('user_1', '2026-03-01 10:00:00', 20),
('user_1', '2026-03-01 10:01:00', 22),
('user_1', '2026-03-01 10:02:00', 21),
('user_1', '2026-03-01 10:03:00', 23),
('user_1', '2026-03-01 10:04:00', 20),

-- User 2: Experiencing a sudden lag spike
('user_2', '2026-03-01 10:00:00', 30),
('user_2', '2026-03-01 10:01:00', 32),
('user_2', '2026-03-01 10:02:00', 150), -- Spike begins here
('user_2', '2026-03-01 10:03:00', 155),
('user_2', '2026-03-01 10:04:00', 31);

Running query in 'sqlite:///advanced_sql.db'

10 rows affected.

++
||
++
++

In [19]:
%%sql
WITH PingDifferences AS (
    SELECT 
        log_id,
        user_id,
        timestamp,
        ping_ms,
        LAG(ping_ms) OVER(
            PARTITION BY user_id 
            ORDER BY timestamp
        ) AS previous_ping_ms,
        ping_ms - LAG(ping_ms) OVER(
            PARTITION BY user_id 
            ORDER BY timestamp
        ) AS ping_spike_amount
    FROM session_telemetry
)
SELECT * FROM PingDifferences
WHERE ping_spike_amount > 50;

Running query in 'sqlite:///advanced_sql.db'

log_id,user_id,timestamp,ping_ms,previous_ping_ms,ping_spike_amount
None,user_2,2026-03-01 10:02:00,150,32,118


In [23]:
%%sql
WITH PingDifferences AS (
    SELECT 
        log_id,
        user_id,
        timestamp,
        ping_ms,
        LAG(ping_ms,2) OVER(
            PARTITION BY user_id 
            ORDER BY timestamp
        ) AS previous_ping_ms,
        ping_ms - LAG(ping_ms, 2) OVER(
            PARTITION BY user_id 
            ORDER BY timestamp
        ) AS ping_spike_amount
    FROM session_telemetry
)
SELECT * FROM PingDifferences
WHERE ping_spike_amount > 50;

Running query in 'sqlite:///advanced_sql.db'

log_id,user_id,timestamp,ping_ms,previous_ping_ms,ping_spike_amount
None,user_2,2026-03-01 10:02:00,150,30,120
None,user_2,2026-03-01 10:03:00,155,32,123


In [33]:
%%sql
DROP TABLE IF EXISTS cluster_sessions;

CREATE TABLE cluster_sessions (
    session_id SERIAL PRIMARY KEY,
    region TEXT,
    user_id VARCHAR(50),
    duration_minutes INTEGER
);

INSERT INTO cluster_sessions (region, user_id, duration_minutes) VALUES
-- US-West: Notice the tie for the longest duration (120 minutes)
('US-West', 'user_w1', 120),
('US-West', 'user_w2', 120), 
('US-West', 'user_w3', 90),
('US-West', 'user_w4', 60),

-- US-East
('US-East', 'user_e1', 150),
('US-East', 'user_e2', 110),
('US-East', 'user_e3', 80),
('US-East', 'user_e4', 45),

-- EU-Central
('EU-Central', 'user_c1', 200),
('EU-Central', 'user_c2', 180),
('EU-Central', 'user_c3', 100),
('EU-Central', 'user_c4', 50);

Running query in 'sqlite:///advanced_sql.db'

12 rows affected.

++
||
++
++

In [38]:
%%sql
select * from cluster_sessions;

Running query in 'sqlite:///advanced_sql.db'

session_id,region,user_id,duration_minutes
None,US-West,user_w1,120
None,US-West,user_w2,120
None,US-West,user_w3,90
None,US-West,user_w4,60
None,US-East,user_e1,150
None,US-East,user_e2,110
None,US-East,user_e3,80
None,US-East,user_e4,45
None,EU-Central,user_c1,200
None,EU-Central,user_c2,180


In [42]:
%%sql
WITH RankedSessions AS (
    SELECT 
        session_id,
        region,
        user_id,
        duration_minutes,
        DENSE_RANK() OVER(
            PARTITION BY region 
            ORDER BY duration_minutes DESC
        ) AS rank_num
    FROM cluster_sessions
)
SELECT * FROM RankedSessions
WHERE rank_num <= 2;

Running query in 'sqlite:///advanced_sql.db'

session_id,region,user_id,duration_minutes,rank_num
None,EU-Central,user_c1,200,1
None,EU-Central,user_c2,180,2
None,US-East,user_e1,150,1
None,US-East,user_e2,110,2
None,US-West,user_w1,120,1
None,US-West,user_w2,120,1
None,US-West,user_w3,90,2
